# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Guest lecture: Padma Venkatraman | Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Oct 1, 2026 - Day 12 - Bayes and Markov chain Monte Carlo (MCMC): from a posterior to samples of it

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

## Today's plan

* today's question: a foreground galaxy splits the light of the quasar RX J1131-1231 into four images, A to D, whose light takes different paths and so arrives at different times
    * **how many days after image A does image D arrive, and how sure are we?**
* Bayes' theorem and priors
    * Day 10's population was really a prior; today we use that idea
* the negative-parallax star from Day 5 finally gets a distance
* why a grid stops working, and the Metropolis algorithm on that 1-D problem
* RX J1131-1231 and its time delay
    * checking a chain: trace plots and burn in, step size, the effective number of samples
    * summarizing, checking the fit, multiple modes, and `emcee`
* from a delay to the Hubble constant
* Lab 05 posts today, due Wed Oct 7 by Noon

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import emcee, corner
from scipy.interpolate import BSpline

from ipywidgets import interact, interactive_output, FloatSlider, IntSlider, FloatLogSlider, HBox, VBox, Layout
from IPython.display import display

# The slow results (the samplers, and every position of the two sampler sliders) are saved in data/day12_cache.npz,
# so Run All takes seconds. Set RECOMPUTE = True to redo them from scratch (a few minutes) and rewrite the file.
import os
CACHE_FILE, RECOMPUTE = 'data/day12_cache.npz', False
_cache = {} if RECOMPUTE or not os.path.exists(CACHE_FILE) else dict(np.load(CACHE_FILE, allow_pickle=False))
def cached(name, compute):
    # compute() returns a dict of arrays; they are stored as name/key, loaded back on the next run
    keys = [k for k in _cache if k.startswith(name + '/')]
    if not keys:
        for k, v in compute().items():
            _cache[f'{name}/{k}'] = np.asarray(v)
        np.savez_compressed(CACHE_FILE, **_cache)
        keys = [k for k in _cache if k.startswith(name + '/')]
    return {k.split('/', 1)[1]: _cache[k] for k in keys}

def controls_right(fn, **sliders):
    # the plot on the left, its sliders stacked on the right
    for w in sliders.values():
        w.layout = Layout(width='230px'); w.style = {'description_width': '75px'}
    display(HBox([interactive_output(fn, sliders), VBox(list(sliders.values()), layout=Layout(justify_content='center', width='240px'))]))

# matplotlib's math-text parser is a shared singleton and is not thread-safe. ipykernel 7 handles slider messages on
# another thread (ipykernel #1564), so a redraw can overlap the next cell and break any "$...$" label
# ("Expected end of text", ipympl #610). A lock around the parser fixes that on any kernel version (ipympl PR #621).
import threading
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

def load_csv(path):
    with open(path) as _f:
        _nhead = sum(1 for _line in _f if _line.startswith('#'))
    return np.genfromtxt(path, delimiter=',', skip_header=_nhead, names=True, dtype=None, encoding='utf-8')

def metropolis(log_post, theta0, step, n_steps, seed=1):
    # The whole algorithm. `step` is the width of the symmetric Gaussian jump, one entry per parameter.
    rng = np.random.default_rng(seed)
    theta = np.array(theta0, dtype=float)
    logp = log_post(theta)
    chain = np.zeros((n_steps, theta.size)); n_accept = 0
    for i in range(n_steps):
        proposal = theta + rng.normal(0, step, size=theta.size)      # propose
        logp_new = log_post(proposal)
        if np.log(rng.uniform()) < logp_new - logp:                   # accept if u < R, with R = P(theta')/P(theta)
            theta, logp = proposal, logp_new
            n_accept += 1
        chain[i] = theta                                              # record either way
    return chain, n_accept / n_steps

def acf(x, max_lag):
    x = x - x.mean()
    return np.array([1.0 if k == 0 else np.corrcoef(x[:-k], x[k:])[0, 1] for k in range(max_lag)])

lc = load_csv('data/rxj1131_cosmograil_millon2020.csv')
recent = lc['hjd'] > 56200                       # the last six observing seasons, 2013 to 2018
t_lc = lc['hjd'][recent]; mA, eA = lc['mag_A'][recent], lc['err_A'][recent]; mD, eD = lc['mag_D'][recent], lc['err_D'][recent]
N_LC = t_lc.size
T0, T1 = t_lc.min() - 250, t_lc.max() + 50
KN = np.r_[[T0] * 3, np.arange(T0, T1, 20.0), [T1] * 4]                                    # quasar curve: a knot every 20 d
KM = np.r_[[t_lc.min() - 1] * 3, np.arange(t_lc.min() - 1, t_lc.max() + 1, 300.0), [t_lc.max() + 1] * 4]   # microlensing of D: every 300 d
B_A = BSpline.design_matrix(t_lc, KN, 3).toarray(); B_ML = BSpline.design_matrix(t_lc, KM, 3).toarray()
N_I, N_M = B_A.shape[1], B_ML.shape[1]

def lens_fit(dt, s):
    # For a given delay dt and jitter s, solve the linear part exactly (Day 9's normal equations):
    # A(t) = q(t),  D(t) = q(t - dt) + microlensing(t),  q and microlensing are splines
    B_D = BSpline.design_matrix(np.clip(t_lc - dt, T0, T1 - 1e-6), KN, 3).toarray()
    wA, wD = 1 / np.sqrt(eA**2 + s**2), 1 / np.sqrt(eD**2 + s**2)
    X = np.zeros((2 * N_LC, N_I + N_M)); X[:N_LC, :N_I] = B_A * wA[:, None]
    X[N_LC:, :N_I] = B_D * wD[:, None]; X[N_LC:, N_I:] = B_ML * wD[:, None]
    y = np.r_[mA * wA, mD * wD]
    c = np.linalg.solve(X.T @ X + 1e-8 * np.eye(N_I + N_M), X.T @ y)
    model_A = B_A @ c[:N_I]; model_D = B_D @ c[:N_I] + B_ML @ c[N_I:]
    return c, wA, wD, X, y, model_A, model_D, B_D

def log_post_lens(theta):
    dt, ln_s = theta
    if not (0 < dt < 400 and -8 < ln_s < -1):        # flat priors: D lags A by 0 to 400 d; jitter between 0.3 and 370 mmag
        return -np.inf
    c, wA, wD, X, y = lens_fit(dt, np.exp(ln_s))[:5]
    r = y - X @ c
    return -0.5 * r @ r + np.sum(np.log(wA)) + np.sum(np.log(wD))


# RX J1131-1231 time-delay model (the lens posterior used from the "Sampling the delay" slide on)
lc = load_csv('data/rxj1131_cosmograil_millon2020.csv')
recent = lc['hjd'] > 56200                       # the last six observing seasons, Dec 2012 to 2018
t_lc = lc['hjd'][recent]; mA, eA = lc['mag_A'][recent], lc['err_A'][recent]; mD, eD = lc['mag_D'][recent], lc['err_D'][recent]
N_LC = t_lc.size
M0 = np.median(mA)                               # magnitudes are measured from here, so a prior centred on 0 is sensible
T0, T1 = t_lc.min() - 250, t_lc.max() + 50
KN = np.r_[[T0] * 3, np.arange(T0, T1, 20.0), [T1] * 4]                                    # quasar curve q: a knot every 20 d
KM = np.r_[[t_lc.min() - 1] * 3, np.arange(t_lc.min() - 1, t_lc.max() + 1, 300.0), [t_lc.max() + 1] * 4]   # microlensing m: every 300 d
B_A = BSpline.design_matrix(t_lc, KN, 3).toarray(); B_ML = BSpline.design_matrix(t_lc, KM, 3).toarray()
N_I, N_M = B_A.shape[1], B_ML.shape[1]
SIG_W = 1.0                                      # prior on every spline weight: Gaussian, width 1 mag

def lens_fit(dt, s):
    # A(t) = q(t),  D(t) = q(t - dt) + m(t).  q and m are splines: sums of fixed smooth bumps with unknown weights.
    # For a given delay dt and jitter s the weights enter linearly, so they can be integrated out exactly (Day 9's algebra, Day 10's idea).
    B_D = BSpline.design_matrix(np.clip(t_lc - dt, T0, T1 - 1e-6), KN, 3).toarray()
    wA, wD = 1 / np.sqrt(eA**2 + s**2), 1 / np.sqrt(eD**2 + s**2)
    X = np.zeros((2 * N_LC, N_I + N_M)); X[:N_LC, :N_I] = B_A * wA[:, None]
    X[N_LC:, :N_I] = B_D * wD[:, None]; X[N_LC:, N_I:] = B_ML * wD[:, None]
    y = np.r_[(mA - M0) * wA, (mD - M0) * wD]
    Aprec = X.T @ X + np.eye(N_I + N_M) / SIG_W**2
    c = np.linalg.solve(Aprec, X.T @ y)
    model_A = M0 + B_A @ c[:N_I]; model_D = M0 + B_D @ c[:N_I] + B_ML @ c[N_I:]
    return dict(c=c, wA=wA, wD=wD, X=X, y=y, Aprec=Aprec, model_A=model_A, model_D=model_D, B_D=B_D)

def log_post_lens(theta):
    dt, ln_s = theta
    if not (0 < dt < 400 and -8 < ln_s < -1):        # flat priors: D arrives 0 to 400 d after A; jitter between 0.3 and 370 mmag
        return -np.inf
    f = lens_fit(dt, np.exp(ln_s))
    r = f['y'] - f['X'] @ f['c']
    logdet = np.linalg.slogdet(f['Aprec'])[1]
    # log of the likelihood with the 131 spline weights integrated out (marginalized) under their Gaussian prior
    return (-0.5 * (r @ r + f['c'] @ f['c'] / SIG_W**2) + np.sum(np.log(f['wA'])) + np.sum(np.log(f['wD']))
            - 0.5 * logdet - (N_I + N_M) * np.log(SIG_W))

def chi2_lens(dt, s):
    f = lens_fit(dt, s); r = f['y'] - f['X'] @ f['c']; return r @ r

print(f"RX J1131-1231: {N_LC} epochs in {1 + np.sum(np.diff(t_lc) > 60)} seasons; median errors A {np.median(eA)*1000:.0f} mmag, D {np.median(eD)*1000:.0f} mmag")

## Recap

* every fit so far maximized a likelihood $\mathcal{L}(D\mid\theta)$
    * the probability of the data $D$, given parameters $\theta$
* the answer was a point $\hat\theta$ and a curvature error bar
    * that error bar is right only when the likelihood is one Gaussian-ish peak
* Day 10 built a **forward model**, the steps from parameters to data
    * draw a true mass, put it on the line, add scatter, add measurement error
    * the likelihood is the probability of our data under that forward model
    * its population Gaussian $\mathcal{N}(\mu, \sigma_{\rm pop})$ on the true masses was already a prior

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**What we want is the probability of the parameters, given the data we observed.**

</div>

## Bayes' theorem

* Day 2's rule, $P(H\mid D) = P(D\mid H)\,P(H)/P(D)$, with the hypothesis $H$ now a set of parameter values $\theta$:

$$P(\theta \mid D) = \frac{P(D \mid \theta)\, P(\theta)}{P(D)}, \qquad P(D) = \int P(D\mid\theta')\,P(\theta')\,d\theta'$$

* $P(D\mid\theta)$: the **likelihood**, the same function we have maximized all semester
* $P(\theta)$: the **prior**, the probability of the parameters *without* any data
* $P(\theta\mid D)$: the **posterior**, a whole distribution
* $P(D)$: the **evidence**, the denominator
    * a normalization today; in model comparison it must be known (the Bayes factor, Tue Oct 6)

In [ ]:
# Prior, likelihood and posterior on one theta axis, with sliders for the prior and the likelihood (a "skip" cell)
def plot_three_curves(prior_width=2.5, like_centre=3.0, like_width=0.9):
    th = np.linspace(-6, 8, 700)
    prior = np.exp(-0.5 * (th / prior_width)**2); like = np.exp(-0.5 * ((th - like_centre) / like_width)**2)
    post = prior * like
    fig, ax = plt.subplots(figsize=(6.6, 3.4))
    ax.plot(th, prior / prior.max(), 'C0', ls='--', lw=2, label=r'prior $P(\theta)$')
    ax.plot(th, like / like.max(), 'C2', ls=':', lw=2.2, label=r'likelihood $P(D\mid\theta)$')
    ax.plot(th, post / post.max(), 'C3', lw=2.6, label=r'posterior $\propto$ prior $\times$ likelihood')
    ax.axvline(th[np.argmax(post)], color='C3', lw=0.8)
    ax.set_yticks([]); ax.set_xlabel(r'$\theta$'); ax.legend(frameon=False, fontsize=11, loc='upper left')
    ax.set_title(f'posterior peak at {th[np.argmax(post)]:.2f}; the likelihood alone peaks at {like_centre:.2f}', fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
# the posterior (solid) is the prior (dashed) times the likelihood (dotted); sliders on the right
controls_right(plot_three_curves, prior_width=FloatSlider(value=2.5, min=0.3, max=8.0, step=0.1, description='prior width'), like_centre=FloatSlider(value=3.0, min=-4.0, max=6.0, step=0.1, description='data peak'), like_width=FloatSlider(value=0.9, min=0.2, max=4.0, step=0.1, description='data width'))

* multiply the prior and the likelihood at each $\theta$, then renormalize
* if the likelihood is much narrower than the prior, the posterior follows the likelihood
* then Bayes agrees with the MLE
* **multiply the likelihood by a prior, even a flat one, and you get a probability**

## How to pick a prior

* **the prior is a subjective choice which we get to make!**
* informative priors are based on existing information
    * including previously obtained data, but not the data considered right now
    * e.g. a parallax is positive, a mass is not negative, a period exceeds the exposure
* Gautham: "It is precisely when we do not have enough data that we rely on the prior"
* different priors give different posteriors from the same data, so state your prior
* **maximum likelihood is maximum a posteriori (MAP) with a flat prior: every fit before today is the special case**
* first use: the star Day 5 could not place

## Day 5, in one slide

<img src="images/gaia_spacecraft.jpg" alt="Artist impression of the Gaia spacecraft in front of the Gaia all-sky image of the Milky Way" style="max-height:230px; display:block; margin:0 auto;">

<div style="font-size:0.75em; text-align:center; color:#666;">Spacecraft: ESA/ATG medialab; Milky Way: ESA/Gaia/DPAC; acknowledgement A. Moitinho. CC BY-SA 3.0 IGO.</div>

* Gaia watches a star's position shift as the Earth orbits the Sun: half that wobble is the **parallax** $\varpi$
    * distance $d = 1/\varpi$: 1 mas is 1 kpc
* Gaia never measures $\varpi$ directly: it is one coefficient of a five-parameter fit
    * so noise can push the fitted $\hat\varpi$ below zero
* $1/\hat\varpi$ is a biased estimator: past about 20% fractional error it stops working
* the star we left hanging: $\hat\varpi = -0.3 \pm 0.5$ mas
    * no distance from $1/\hat\varpi$, and "the honest fix needs a prior" (Day 5)

## The negative-parallax star

<img src="images/bailerjones_distance_from_parallax.png" alt="Bailer-Jones 2015 Figure 2: bias and scatter of the one-over-parallax distance estimate against fractional parallax error" style="max-height:260px; display:block; margin:0 auto;">

<div style="font-size:0.75em; text-align:center; color:#666;">Bailer-Jones 2015, PASP 127, 994, Fig. 2, the Day 5 figure.</div>

* our star: $\hat\varpi = -0.3 \pm 0.5$ mas, and no maximum-likelihood **distance** exists
* write the distance as $r$ (the $d$ of the last slide)
* the prior, for stars spread through the Galaxy: $p(r) \propto r^2 e^{-r/L}$
    * $L = 1.35$ kpc (Bailer-Jones 2015; Astraatmadja & Bailer-Jones 2016)
* the likelihood is the same Gaussian in $\varpi$, evaluated at $\varpi = 1/r$

In [ ]:
# The negative-parallax posterior on a grid, with sliders for the measurement and the prior scale (a "skip" cell)
r_grid = np.linspace(0.01, 20, 4000)
def parallax_posterior(varpi_hat=-0.3, sig_varpi=0.5, L=1.35):
    like = np.exp(-0.5 * ((1.0 / r_grid - varpi_hat) / sig_varpi)**2)
    prior = r_grid**2 * np.exp(-r_grid / L)
    post = like * prior; post /= np.trapezoid(post, r_grid)
    cdf = np.cumsum(post) * (r_grid[1] - r_grid[0])
    return like, prior / np.trapezoid(prior, r_grid), post, r_grid[np.argmax(post)], np.interp([0.16, 0.5, 0.84], cdf, r_grid)

def log_post_r(r, varpi_hat=-0.3, sig_varpi=0.5, L=1.35):
    r = float(np.squeeze(r))
    if r <= 0:
        return -np.inf
    return -0.5 * ((1.0 / r - varpi_hat) / sig_varpi)**2 + 2 * np.log(r) - r / L      # log likelihood + log prior

_, _, post_r, r_mode, (r16, r50, r84) = parallax_posterior()

def plot_parallax(varpi_hat=-0.3, sig_varpi=0.5, L=1.35):
    like, prior, post, mode, (lo, med, hi) = parallax_posterior(varpi_hat, sig_varpi, L)
    fig, ax = plt.subplots(figsize=(6.6, 3.4))
    ax.plot(r_grid, prior, 'C0', ls='--', lw=1.8, label=r'prior $p(r) \propto r^2 e^{-r/L}$')
    ax.plot(r_grid, like / like.max() * post.max(), 'C2', ls=':', lw=2, label=r'likelihood at $\varpi = 1/r$ (scaled)')
    ax.plot(r_grid, post, 'C3', lw=2.6, label='posterior')
    ax.axvspan(lo, hi, color='C3', alpha=0.15, label=f'mode {mode:.1f} kpc, 68%: {lo:.1f} to {hi:.1f} kpc')
    ax.set_xlabel('distance $r$ [kpc]'); ax.set_yticks([]); ax.set_xlim(0, 15); ax.legend(frameon=False, fontsize=10, loc='upper right')
    plt.tight_layout(); plt.show()

* move the measured parallax, its error bar, and the prior's scale $L$

In [ ]:
# prior (dashed), likelihood mapped to distance (dotted), posterior (solid); the band holds 68% of the posterior
controls_right(plot_parallax, varpi_hat=FloatSlider(value=-0.3, min=-1.0, max=2.0, step=0.1, description='ϖ̂ [mas]'), sig_varpi=FloatSlider(value=0.5, min=0.05, max=1.5, step=0.05, description='σ_ϖ [mas]'), L=FloatSlider(value=1.35, min=0.3, max=4.0, step=0.05, description='L [kpc]'))

* at $-0.3 \pm 0.5$ mas the posterior peaks near 3.6 kpc and is wide
* with data this weak, the answer depends on the prior
* shrink the error bar and the likelihood takes over
* this is a **grid**: 4000 values of $r$, evaluate the posterior at each, normalize

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**It is often true that Bayesian analysis and traditional MLE are essentially equivalent. However, in some cases, considering the priors can have significant consequences.**

</div>

## Curse of dimensionality

* 100 points per parameter is $100^k$ posterior evaluations for $k$ parameters
* $k = 1$: 100. $k = 7$ (Lab 05's orbit): $10^{14}$, about 30 years at 10 microseconds each
* the lensed quasar later today has over a hundred unknowns; we will see how to handle them
* keep sampling in regions of high probability, don't sample in regions with low probability

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We want to focus resources on mapping the posterior where it is non-tiny. Generating samples from the posterior itself automatically accomplishes this.**

</div>

## The Metropolis algorithm

1. start somewhere, $\theta_0$
2. **propose** a nearby point $\theta'$: a random jump from a symmetric Gaussian of width `step`
3. compute the ratio $R = P(\theta'\mid D)\,/\,P(\theta_i\mid D)$
4. **accept** $\theta'$ if $R \ge 1$; otherwise accept it with probability $R$
5. **record** a sample either way (the old point again if you rejected), and repeat

* the evidence $P(D)$ cancels in $R$: we only ever need $\log(\text{prior}) + \log(\text{likelihood})$
* the history: Metropolis et al. 1953, at Los Alamos ([the LANL account](https://permalink.lanl.gov/object/tr?what=info:lanl-repo/lareport/LA-UR-88-9068))

## Why it works

* every $(n+1)$th position depends **only** on the $n$th: chains like this are called **Markov chains**
    * the random jumps make it **Monte Carlo**
* the stochasticity is the point!
    * it allows us to explore the surface, but spend more time in interesting spots
    * even if a step is rejected, we still keep a sample - the difficulty of finding a temptingly better point is important information!
* why the time spent matches the posterior: Gautham derives it on Oct 6
* Hastings (1970) added a correction for asymmetric jumps: **Metropolis-Hastings**
* play with it: [a Markov chain, step by step](http://setosa.io/ev/markov-chains/) and [the MCMC demo gallery](http://chi-feng.github.io/mcmc-demo/)

In [ ]:
# Metropolis on the 1-D parallax posterior: 5000 iterations in r, then compare the sample histogram with the grid (a "skip" cell)
chain_r, acc_r = metropolis(log_post_r, [1.0], [5.0], 5000, seed=7)

def plot_metropolis_1d():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw=dict(width_ratios=[1.3, 1]))
    a1.plot(chain_r[:600, 0], color='C0', lw=0.8); a1.set_xlabel('iteration'); a1.set_ylabel('$r$ [kpc]'); a1.set_title('the first 600 iterations of the chain', fontsize=11)
    a2.hist(chain_r[:, 0], bins=60, density=True, color='C0', alpha=0.6, label=f'{chain_r.shape[0]} Metropolis samples')
    a2.plot(r_grid, post_r, 'k', lw=2.2, label='the grid posterior')
    a2.set_xlabel('$r$ [kpc]'); a2.set_yticks([]); a2.set_xlim(0, 20); a2.legend(frameon=False, fontsize=11)
    plt.tight_layout(); plt.show()
    print(f"acceptance fraction {acc_r:.2f} (the fraction of proposals accepted); sample median {np.median(chain_r):.2f} kpc vs grid median {r50:.2f} kpc")

## Metropolis on the parallax star

* the same 1-D posterior, now sampled: 5000 iterations, jumps of width 5 kpc

In [ ]:
# left: the chain wandering in r; right: where it spent its time, against the grid posterior it never saw whole
plot_metropolis_1d()

* the **acceptance fraction** is the fraction of proposals accepted
* the chain never used $P(D)$ or the grid; the grid curve is only there for comparison
* next: a real measurement, with two parameters and a trap

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The histogram of the samples is the posterior: time spent at r is the posterior at r.**

</div>

## RX J1131-1231

<img src="images/rxj1131_hubble_heic1702d_labelled.png" alt="Hubble image of RX J1131-1231: the lens galaxy G at the centre, three bright quasar images B, A and C on an arc to the left, and a fainter fourth image D to the right, all on a ring of stretched host-galaxy light" style="max-height:330px; display:block; margin:0 auto;">

<div style="font-size:0.75em; text-align:center; color:#666;">ESA/Hubble, NASA, Suyu et al. (heic1702d, CC BY 4.0); labels A-D and lens galaxy G added, positions from Suyu et al. 2013, Fig. 1.</div>

* a **quasar** is a galaxy's central black hole, swallowing gas and outshining the galaxy
    * its brightness flickers over days to months
* a galaxy (G) lies almost exactly in front of it and bends its light into four images, A to D, and a ring
* **today's question: how many days after image A does image D arrive, and how sure are we?**

In [ ]:
# A cartoon of the lens geometry, over a grid distorted by the lens galaxy's gravity (a "skip" cell)
def plot_lens_cartoon():
    fig, ax = plt.subplots(figsize=(10, 3.6))
    xs, xl, xo = 0.0, 5.0, 10.0
    # background grid, pulled toward the lens galaxy: each point moves toward (xl, 0) by theta_E^2 / distance
    thE2 = 0.6
    for const in np.arange(-1.4, 1.6, 0.25):
        for horizontal in (True, False):
            u = np.linspace(-0.8, 10.8, 400) if horizontal else np.linspace(-1.4, 1.5, 200)
            px, py = (u, np.full_like(u, const)) if horizontal else (np.full_like(u, xl + 4 * const), u)
            dx, dy = px - xl, py; r2 = dx**2 + dy**2 + 0.05
            pull = np.minimum(thE2 / r2, 0.8)
            ax.plot(px - pull * dx, py - pull * dy, color='0.85', lw=0.7, zorder=0)
    ax.plot(xs, 0, marker='*', ms=22, color='C1'); ax.text(xs, -0.6, 'quasar', ha='center', fontsize=12)
    ax.plot(xl, 0, marker='o', ms=26, color='0.5'); ax.text(xl, 0.3, 'lens galaxy', ha='center', fontsize=12)
    ax.plot(xo, 0, marker='o', ms=10, color='k'); ax.text(xo, -0.6, 'us', ha='center', fontsize=12)
    ax.plot([xs, xl, xo], [0, 1.0, 0], color='C0', lw=2); ax.text(xl, 1.15, 'light forming image A: arrives first', ha='center', fontsize=12, color='C0')
    ax.plot([xs, xl, xo], [0, -0.35, 0], color='C3', lw=2); ax.text(xl, -1.05, 'light forming image D: arrives $\\Delta t$ later', ha='center', fontsize=12, color='C3')
    ax.set_xlim(-0.8, 10.8); ax.set_ylim(-1.3, 1.5); ax.axis('off')
    plt.tight_layout(); plt.show()

## Why a delay measures the Hubble constant

In [ ]:
# not to scale: two of the four light paths
plot_lens_cartoon()

* the paths differ in length and in how deep they pass through G's gravity
    * so image D shows the quasar $\Delta t$ days after image A
* $\Delta t$ scales with the distances to the galaxy and the quasar
* every distance scales as $1/H_0$, the **Hubble constant** (Day 9: Hubble's diagram)
* so with a model of how the galaxy's mass is distributed (its **mass model**) fixed, $H_0 \propto 1/\Delta t$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Measure the delay, model the lens, and you have the Hubble constant.**

</div>

In [ ]:
# The two light curves, with D moved back by a trial delay, and the misfit at that delay (a "skip" cell)
S_BEST = 0.0245                                     # the jitter at the posterior peak, in mag
def plot_shift(dt=0.0):
    off = np.median(mD - mA)
    fig, ax = plt.subplots(figsize=(7.0, 3.4))
    ax.errorbar(t_lc, mA, yerr=eA, fmt='o', ms=3, color='C0', label='image A')
    ax.errorbar(t_lc - dt, mD - off, yerr=eD, fmt='o', ms=3, color='C1', label=f'image D, moved back {dt:.0f} d')
    ax.set_xlabel('HJD - 2400000 [d]'); ax.set_ylabel('relative magnitude'); ax.invert_yaxis(); ax.legend(frameon=False, fontsize=11, loc='lower left')
    ax.set_title(f'misfit at this delay (lower is better): {chi2_lens(dt, S_BEST):.0f} for {2 * N_LC} points', fontsize=12)
    plt.tight_layout(); plt.show()

## Two light curves, one quasar

* COSMOGRAIL (COSmological MOnitoring of GRAvItational Lenses) has watched it since 2003 (Millon et al. 2020)
* we use images A and D from six seasons, late 2012 to 2018
* times are heliocentric Julian dates (HJD), a running count of days
* drag image D back in time until its wiggles fall on image A's
    * the number in the plot title scores the match (lower is better); the next slides say how it is computed

In [ ]:
# image A (blue) and image D (orange), with D moved back by the slider; the title is the misfit at that delay
controls_right(plot_shift, dt=FloatSlider(value=0.0, min=0.0, max=300.0, step=1.0, continuous_update=False, description='Δt [d]'))

* image D looks like image A shifted later by about 95 days
* image D also drifts slowly against image A: **microlensing**
    * stars in the lens galaxy magnify image D's light by amounts that change over the years
* image D is fainter, so its error bars are 28 mmag against image A's 4

## Sampling the delay

* move image D back by $\Delta t$, fit one smooth curve to both images, and score it
* the model, for the light curves $y_A$ and $y_D$ of images A and D: $y_A(t) = q(t)$ and $y_D(t) = q(t - \Delta t) + m(t)$
* $q$ is the quasar's own light curve, and $m$ is image D's microlensing
* both are **splines**, sums of fixed smooth bumps with 131 unknown weights in all
* for any $\Delta t$ the weights enter linearly
    * so we integrate them out exactly (Day 9's algebra, Day 10's idea)

## What we sample

* the delay $\Delta t$, and a **jitter** $s$ added to every error bar in quadrature
    * $\sigma_i^2 \to \sigma_i^2 + s^2$, like Day 10's intrinsic scatter
    * what we've done is expanded our model with a **nuisance parameter**, which we marginalize out at the end
* priors: flat in $\Delta t$ from 0 to 400 d, and flat in $\ln s$
    * each spline weight gets a Gaussian prior of width 1 mag
* start at $\Delta t = 80$ d and run 10,000 iterations
* step sizes: 1.2 times each curvature error bar at the peak, 1.55 d in $\Delta t$ and 0.082 in $\ln s$

In [ ]:
# Run Metropolis on the time-delay posterior (a "skip" cell)
theta0_lens = [80.0, -3.0]
step_lens = np.array([1.547, 0.082])                 # 1.2 x each parameter's curvature error bar at the peak
def _run_lens():
    chain, acc = metropolis(log_post_lens, theta0_lens, step_lens, 10_000, seed=1)
    return dict(chain=chain, acc=acc)
_r = cached('lens', _run_lens); chain_lens, acc_lens = _r['chain'], float(_r['acc'])
burn = 300                                           # read off the trace plot: both settle within ~100 iterations
post_lens = chain_lens[burn:]
med_lens = np.median(post_lens, axis=0)
NAMES = (r'$\Delta t$ [d]', r'$\ln s$')

def report_lens():
    print(f"acceptance fraction {acc_lens:.2f}")
    print(f"posterior medians (first 300 iterations dropped; the next slide says why): delay = {med_lens[0]:.1f} d, jitter s = {np.exp(med_lens[1])*1000:.1f} mmag")

In [ ]:
# 10,000 iterations (about seven seconds; loaded from the saved run); the acceptance fraction and the posterior medians
report_lens()

* about half of the proposals were accepted
* "found the peak" and "finished exploring it" are different claims
* trace plots and the autocorrelation time tell them apart

## Trace plots and burn in

* a **trace plot** is the chain's value against iteration number, one panel per parameter
* the first iterations, before the chain reaches the high-posterior region, are not posterior samples
* this period is called the **burn in**, and we discard it
* there are numerical estimates that can help with this
* but **they are not a substitute for human visual inspection**

In [ ]:
# Trace plot of the chain, with a zoom on the start (a "skip" cell)
def plot_trace(chain, burn, names=NAMES, medians=None):
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 4.6), gridspec_kw=dict(width_ratios=[2.2, 1]))
    for i, name in enumerate(names):
        for ax, n in zip(axes[i], (chain.shape[0], 600)):
            ax.plot(np.arange(n), chain[:n, i], lw=0.6, color='C0')
            ax.axvspan(0, burn, color='0.8'); ax.axvline(burn, color='k', lw=0.8)
            if medians is not None: ax.axhline(medians[i], color='C3', ls='--', lw=1)
        axes[i, 0].set_ylabel(name)
    axes[0, 1].set_title('the first 600 iterations', fontsize=11)
    for ax in axes[1]: ax.set_xlabel('iteration')
    plt.tight_layout(); plt.show()

In [ ]:
# left: all 10,000 iterations; right: the first 600. Grey: the 300 we discard; dashed: the posterior median
plot_trace(chain_lens, burn, medians=med_lens)

* both parameters settle within about 100 iterations of the start at 80 d
* we discard the first 300 to be safe

In [ ]:
# The same chain with the step size scaled by a slider: trace of the delay and the acceptance fraction (a "skip" cell)
STEP_EXPONENTS = np.round(np.arange(-1.3, 1.31, 0.1), 1)          # every position of the log slider
def _run_steps():
    out = {}
    for e in STEP_EXPONENTS:
        ch, a = metropolis(log_post_lens, theta0_lens, step_lens * 10**e, 3000, seed=2)
        out[f'dt_{e:+.1f}'], out[f'acc_{e:+.1f}'] = ch[:, 0], a
    return out
step_runs = cached('steps', _run_steps)

def plot_step(scale=1.0):
    e = np.round(np.log10(scale), 1); dt_trace, a = step_runs[f'dt_{e:+.1f}'], float(step_runs[f'acc_{e:+.1f}'])
    fig, ax = plt.subplots(figsize=(6.6, 3.2))
    ax.plot(dt_trace, lw=0.7, color='C0'); ax.axhline(med_lens[0], color='C3', ls='--', lw=1)
    ax.set_xlabel('iteration'); ax.set_ylabel(r'$\Delta t$ [d]'); ax.set_ylim(75, 110)
    ax.set_title(f'step size x {scale:.2g}: {a:.0%} of proposals accepted', fontsize=12)
    plt.tight_layout(); plt.show()

## Choosing the step size

* with Metropolis, you need to specify the step size "just right" in every dimension
* or mixing (how fast the chain covers the posterior) will be slow
* slide the width; each release runs 3000 iterations (a couple of seconds)

In [ ]:
# the delay over 3000 iterations; dashed: the posterior median
controls_right(plot_step, scale=FloatLogSlider(value=1.0, base=10, min=-1.3, max=1.3, step=0.1, continuous_update=False, description='step ×'))

* if the step is too small, we don't explore much of the parameter space
    * ×0.1: 90% accepted, and about 800 iterations to reach the peak
* if it is too big, the chain heads to low-density regions too often
    * ×10: 1% accepted, so nearly every $\theta'$ is rejected
* **monitor the acceptance rate and adjust**
    * the optimum is about 0.44 in 1-D and 0.23 in many dimensions
* try the same on a banana: [random-walk Metropolis, with a width slider](https://chi-feng.github.io/mcmc-demo/app.html?algorithm=RandomWalkMH&target=banana)

## Effective number of samples

* our samples are now correlated - i.e. each one is the last one, adjusted by a small amount
    * so 9,700 Metropolis samples aren't 9,700 samples, because they aren't **independent**
* the **autocorrelation** $\rho_\ell$ at lag $\ell$: the chain correlated with itself shifted by $\ell$ iterations
* the **integrated autocorrelation time** $\tau = 1 + 2\sum_\ell \rho_\ell$
    * roughly how many iterations apart two samples must be to be independent
$$N_{\textrm{eff}} \approx N_{\textrm{samples}} / \tau$$
* Hogg & Foreman-Mackey 2018: the autocorrelation time is the most important test for convergence

In [ ]:
# Autocorrelation of the post-burn-in chain, and emcee's integrated autocorrelation time (a "skip" cell)
tau_int = np.array([emcee.autocorr.integrated_time(post_lens[:, i], quiet=True)[0] for i in range(2)])
def plot_acf():
    fig, ax = plt.subplots(figsize=(8.5, 3.4)); lags = np.arange(60)
    for i, col, lab in [(0, 'C2', r'$\Delta t$'), (1, 'C3', r'$\ln s$')]:
        ax.plot(lags, acf(post_lens[:, i], 60), color=col, lw=1.8, ls='-' if i == 0 else '--', label=f'{lab}: integrated time {tau_int[i]:.0f} iterations')
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xlabel(r'lag $\ell$ [iterations]'); ax.set_ylabel(r'autocorrelation $\rho_\ell$'); ax.legend(frameon=False, fontsize=11)
    plt.tight_layout(); plt.show()
    print(f"{post_lens.shape[0]} post-burn-in samples -> about {post_lens.shape[0] / tau_int.max():.0f} effectively independent ones")

In [ ]:
# the correlation of the chain with a shifted copy of itself
plot_acf()

* $\tau \approx 7$ to 9 iterations here: about 1100 effectively independent samples
* **thinning** keeps every $k$th sample to save space; it does not add information
* the `emcee` package (later today) recommends a chain longer than $50\tau$; ours is over $1000\tau$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The larger lag one needs to get a small autocorrelation, the less informative individual samples are.**

</div>

## Summarizing our inferences

* the marginalized 1D posterior for a parameter is a histogram of that column of the chain
* statistics like the median or percentiles of the posterior are estimated directly from the samples
* we integrate out a nuisance parameter simply by ignoring its column: here, the jitter

In [ ]:
# Corner plot of the post-burn-in chain (a "skip" cell)
def plot_corner():
    fig = corner.corner(post_lens, labels=NAMES, quantiles=[0.16, 0.5, 0.84], show_titles=False, label_kwargs=dict(fontsize=13))
    fig.set_size_inches(5.4, 5.4); plt.show()
    lo, mid, hi = np.percentile(post_lens, [16, 50, 84], axis=0)
    print(f"delay = {mid[0]:.1f} (+{hi[0]-mid[0]:.1f} / -{mid[0]-lo[0]:.1f}) d;  jitter s = {np.exp(mid[1])*1000:.1f} mmag")

In [ ]:
# the corner plot from Day 5, now built from samples: diagonal = one parameter each, off-diagonal = both together
plot_corner()

* six seasons give $\Delta t = 94.6 \pm 1.3$ d
* Millon et al. 2020 find $92.5^{+1.9}_{-1.8}$ d from fifteen seasons
* their microlensing model is more flexible than ours
* the delay and the jitter barely trade off: a round blob

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Image D arrives 94.6 ± 1.3 days after image A - if our model is adequate.**

</div>

## Posterior-predictive checks

* if the model is right, data simulated from it should look like ours
* draw parameters from the chain
* from each, simulate fake light curves with the observed error bars plus the jitter
* compare $\chi^2$ of each fake, and of the real data, against the same model

In [ ]:
# Posterior predictive check: the lined-up curves with one fake D, and chi2 of fakes vs the real data (a "skip" cell)
def _run_ppc():
    rng = np.random.default_rng(11); draws = post_lens[rng.integers(0, post_lens.shape[0], 300)]
    chi_fake, chi_real = [], []
    for dt, ln_s in draws:
        f = lens_fit(dt, np.exp(ln_s)); w = np.r_[f['wA'], f['wD']]; model = np.r_[f['model_A'], f['model_D']]
        fake = model + rng.normal(0, 1 / w)
        chi_fake.append(np.sum(((fake - model) * w)**2)); chi_real.append(np.sum(((np.r_[mA, mD] - model) * w)**2))
    f = lens_fit(med_lens[0], np.exp(med_lens[1])); s = np.exp(med_lens[1])
    return dict(chi_fake=chi_fake, chi_real=chi_real, fakeD=f['model_D'] + rng.normal(0, np.sqrt(eD**2 + s**2)), chi2_nojitter=chi2_lens(med_lens[0], 1e-4))
ppc = cached('ppc', _run_ppc)

def plot_ppc():
    chi_fake, chi_real, fakeD = ppc['chi_fake'], ppc['chi_real'], ppc['fakeD']
    f = lens_fit(med_lens[0], np.exp(med_lens[1])); micro = B_ML @ f['c'][N_I:]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.0), gridspec_kw=dict(width_ratios=[1.6, 1]))
    seasons = np.split(t_lc, np.where(np.diff(t_lc) > 60)[0] + 1)
    for j, ts in enumerate(seasons):                  # draw q only inside seasons: in the gaps nothing constrains it
        tg = np.linspace(ts.min(), ts.max(), 300); q = M0 + BSpline.design_matrix(tg, KN, 3).toarray() @ f['c'][:N_I]
        a1.plot(tg, q, color='k', lw=1, label='$q(t)$, the quasar' if j == 0 else None)
    a1.errorbar(t_lc, mA, yerr=eA, fmt='o', ms=2.5, color='C0', label='image A')
    a1.errorbar(t_lc - med_lens[0], mD - micro, yerr=eD, fmt='o', ms=2.5, color='C1', label=r'image D, moved back, microlensing removed')
    a1.plot(t_lc - med_lens[0], fakeD - micro, 's', ms=2.5, color='0.55', label='one fake D from the posterior')
    a1.set_xlim(t_lc.min(), t_lc.max()); a1.invert_yaxis(); a1.set_xlabel('HJD - 2400000 [d]'); a1.set_ylabel('relative magnitude')
    a1.legend(frameon=False, fontsize=10, ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.17))
    a2.hist(chi_fake, bins=30, color='0.6', label='300 fake datasets'); a2.hist(chi_real, bins=30, color='C0', alpha=0.6, label='the real data')
    a2.set_xlabel(r'$\chi^2$ (' + f'{2 * N_LC} points)'); a2.set_yticks([]); a2.legend(frameon=False, fontsize=11)
    plt.tight_layout(); plt.show()
    r0 = float(ppc['chi2_nojitter'])
    print(f"with the jitter: real chi2 {np.median(chi_real):.0f}, fakes {np.median(chi_fake):.0f} (medians). Without it (s = 0): chi2 {r0:.0f} for {2 * N_LC} points")

In [ ]:
# left: A, and D moved back on one quasar curve, with one fake D in grey; right: chi2 of the fakes (grey) and of the real data (blue)
plot_ppc()

* without the jitter, $\chi^2 \approx 3030$ for 450 points: the error bars alone miss real scatter
* with a 24.5 mmag jitter, real $\chi^2$ (about 459) sits among the fakes (about 449)
* the fakes are not refit, so their $\chi^2$ is about one per point
* if the model is right, you expect to see something similar to what you did the first time
* passing is not proof: a check can only find the misfits it looks for

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Nothing about the Bayesian framework tells us if our model is right. MCMC can give us very precise, but very wrong inferences, if the model itself is inadequate.**

</div>

In [ ]:
# Where the chain goes from a chosen start, next to the posterior along the delay axis (a "skip" cell)
START_VALUES = np.arange(60, 251, 5)                              # every position of the slider
def _run_starts():
    dt_grid = np.arange(60, 260, 1.0)
    prof = np.array([max(log_post_lens([g, l]) for l in np.linspace(-4.2, -2.6, 9)) for g in dt_grid])
    out = dict(dt_grid=dt_grid, prof_grid=prof - prof.max())
    for st in START_VALUES:
        out[f'chain_{st}'] = metropolis(log_post_lens, [float(st), -2.9], step_lens, 3000, seed=3)[0]
    return out
start_runs = cached('starts', _run_starts)
dt_grid, prof_grid = start_runs['dt_grid'], start_runs['prof_grid']

def plot_start(start=215.0):
    ch = start_runs[f'chain_{int(round(start))}']
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.4, 3.3))
    a1.plot(ch[:, 0], lw=0.7, color='C1'); a1.axhline(med_lens[0], color='C0', ls='--', lw=1)
    a1.set_xlabel('iteration'); a1.set_ylabel(r'$\Delta t$ [d]'); a1.set_title(f'start at {start:.0f} d: ends near {np.median(ch[-500:, 0]):.0f} d', fontsize=12)
    a2.plot(dt_grid, prof_grid, color='k'); a2.axvline(med_lens[0], color='C0', ls='--'); a2.axvline(start, color='C1', ls=':')
    a2.set_xlabel(r'$\Delta t$ [d]'); a2.set_ylabel('log posterior, relative to the peak'); a2.set_title('the posterior along the delay axis', fontsize=12)
    plt.tight_layout(); plt.show()
    j = np.argmin(abs(dt_grid - np.median(ch[-500:, 0])))
    print(f"where the chain ended: log posterior {prof_grid[j]:.0f} below the true peak; jitter {np.exp(np.median(ch[-500:, 1]))*1000:.0f} mmag (true peak: {np.exp(med_lens[1])*1000:.0f})")

## Coping with multiple modes

* start the same chain somewhere else: move the start and release

In [ ]:
# left: the delay trace from the chosen start (dashed: the true peak); right: the posterior along the delay axis
controls_right(plot_start, start=FloatSlider(value=215.0, min=60.0, max=250.0, step=5.0, continuous_update=False, description='start [d]'))

* from 215 d the chain stays on a side peak for good
* that peak's log posterior is about 210 below the true peak
* there it needs 60 mmag of jitter to call the misfit noise
* the true peak needs only 25 mmag: the peak that explains the data with less extra noise is the better one
* from 150 d it escapes: a shallow side peak lets go, a deep one doesn't

## Coping with multiple modes: why, and what to do

* to get out you'd have to make a large step size
    * but then your acceptance ratio goes down and your autocorrelation time goes up
* the side peaks come from the gaps between observing seasons
    * Day 5's aliasing, with seasons instead of nights
* Day 7 said sampling would handle aliasing, but sampling explores only its current peak
    * finding the right peak is a search: scan first, as in the right panel of the last slide (the posterior along the delay axis)
    * Lab 05 does this scan with a periodogram
* see it on a toy: [random-walk Metropolis on two peaks](https://chi-feng.github.io/mcmc-demo/app.html?algorithm=RandomWalkMH&target=multimodal)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Multiple, well separated posterior modes are a serious challenge for many samplers. In general, the only way to discover that they exist is by exploring the parameter space with many widely dispersed chains.**

</div>

## The same fit in `emcee`

* tuning is kind of boring and painful
    * if we're going to start multiple chains anyway, might as well start them all at once
* `emcee` runs many chains (**walkers**) together
    * where each walker jumps next depends on where the other walkers are
    * each walker jumps along the line through another walker: the **stretch move** (Goodman & Weare 2010)
* its one knob, the stretch scale $a$, acts like a step size; the default $a = 2$ almost always works

```python
rng = np.random.default_rng(3)
theta_start = med_lens + 1e-3 * step_lens * rng.normal(size=(16, 2))   # 16 walkers in a small ball near the answer
sampler = emcee.EnsembleSampler(nwalkers=16, ndim=2, log_prob_fn=log_post_lens)
sampler.run_mcmc(theta_start, 1600)
tau = sampler.get_autocorr_time()                                         # integrated, per parameter
flat = sampler.get_chain(discard=int(3 * tau.max()), thin=int(tau.max() / 2), flat=True)
```

In [ ]:
# Run emcee on the same posterior, and once more from the side peak (a "skip" cell)
def _run_emcee():
    rng_e = np.random.default_rng(3)
    np.random.seed(12)                              # emcee draws its moves from numpy's global generator: seed it so the result is fixed
    sampler = emcee.EnsembleSampler(16, 2, log_post_lens)
    sampler.run_mcmc(med_lens + 1e-3 * step_lens * rng_e.normal(size=(16, 2)), 1600, progress=False)
    tau = sampler.get_autocorr_time(quiet=True)
    flat = sampler.get_chain(discard=int(3 * tau.max()), thin=max(1, int(tau.max() / 2)), flat=True)
    np.random.seed(13)
    trap = emcee.EnsembleSampler(16, 2, log_post_lens)
    trap.run_mcmc(np.array([215.0, -2.83]) + 1e-3 * step_lens * rng_e.normal(size=(16, 2)), 500, progress=False)
    return dict(tau=tau, flat=flat, acc=sampler.acceptance_fraction.mean(), trap_last=trap.get_chain()[-1, :, 0])
em = cached('emcee', _run_emcee)
tau_e, flat_e, acc_e, dt_trap_last = em['tau'], em['flat'], float(em['acc']), em['trap_last']

def report_emcee():
    print(f"emcee: mean acceptance {acc_e:.2f}; integrated autocorrelation times {np.round(tau_e).astype(int)} iterations; {flat_e.shape[0]} thinned samples")
    print(f"emcee:      delay = {np.median(flat_e[:, 0]):.1f} d, jitter = {np.exp(np.median(flat_e[:, 1]))*1000:.1f} mmag")
    print(f"Metropolis: delay = {med_lens[0]:.1f} d, jitter = {np.exp(med_lens[1])*1000:.1f} mmag")
    print(f"emcee started on the side peak: after 500 iterations all {dt_trap_last.size} walkers are between {dt_trap_last.min():.0f} and {dt_trap_last.max():.0f} d")

In [ ]:
# 16 walkers x 1600 iterations, then 500 more from the side peak (about twenty seconds; loaded from the saved run)
report_emcee()

* same medians as Metropolis - and critically, no tuning step!
* but remember that these are not *independent* chains
    * if the ensemble is not started in a region of high probability, convergence will be **extremely** slow
    * started on the side peak, the walkers stay there and never reach the true peak
    * here we cheated and started at the Metropolis answer; in practice, start from the scan
* acceptance is 0.72; Foreman-Mackey et al. 2013 suggest 0.2 to 0.5
    * an easy 2-D posterior accepts more
* documentation and tutorials: [emcee](https://emcee.readthedocs.io/en/stable/)
    * it's very unlikely you will use plain Metropolis directly for research

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Unlike regular Metropolis-Hastings, you don't have to specify a step size, just the initial positions of the walkers.**

</div>

## From a delay to $H_0$

* how a thousand delays like ours will become $H_0$: the figure reads top to bottom, from simulated lenses to one lens to the whole population

<img src="images/venkatraman2025_fig1_pgm.png" alt="Venkatraman et al. 2026 Figure 1: a probabilistic graphical model with three levels: training on simulated lenses, a neural network posterior for each lens mass model from LSST images, and inference of the population of lenses" style="max-height:380px; display:block; margin:0 auto;">

<div style="font-size:0.75em; text-align:center; color:#666;">Venkatraman et al. 2026, AJ 172, 223 (arXiv:2510.20778), Fig. 1 (CC BY 4.0).</div>

## From a delay to $H_0$

* to get $H_0$ we also need the lens galaxy's mass model
* **LSST**, the NSF-DOE Vera C. Rubin Observatory's Legacy Survey of Space and Time, will find about a thousand lensed quasars
    * too many to model one at a time by hand
* Venkatraman et al. 2026 train a neural network on 500,000 simulated images
    * it returns a posterior for each lens's mass model
    * on about 1300 simulated LSST lenses it recovers the **Einstein radius** (the ring's radius in the RX J1131 image, which sets the lens mass) with under 1% bias and 6.5% precision
* then the whole population is fit at once: Day 10's population idea, one level up
* each lens also needs a delay like the one we just sampled
* **with 800 lenses that have measured delays, Erickson et al. 2026 forecast about 2.5% on $H_0$**

<div style="font-size:0.75em; text-align:center; color:#666;">Forecast: Erickson, Millon, Venkatraman et al. 2026, OJAp (arXiv:2511.13669).</div>

## Wrapping up

* posterior $\propto$ likelihood $\times$ prior
    * **the prior is a subjective choice which we get to make!**
* Metropolis samples where the posterior is big, without ever computing $P(D)$
* minimally, check the trace plot and autocorrelation, and discard burn in
* the numbers are percentiles of the chain
* the posterior-predictive check asks if the model is adequate
* with multiple modes, find the right peak first, then sample it
* each delay like this one, with a lens mass model, gives $H_0$
* MCMC is not a magic bullet: just because you sampled the posterior, doesn't mean your samples are useful

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Image D arrives 94.6 ± 1.3 days after image A: the ± 1.3 is how sure we are, because we checked the burn in, τ, the fit and the start.**

</div>

## Lab 05: the dark companion

<img src="images/elbadry2023_gaiabh1_rv.png" alt="El-Badry et al. 2023 Figure 2: radial velocities of the Gaia BH1 companion star from 2017 to 2023 with posterior draws of an eccentric 186-day orbit, and a zoom on 2022" style="max-height:300px; display:block; margin:0 auto;">

<div style="font-size:0.75em; text-align:center; color:#666;">El-Badry et al. 2023, MNRAS 518, 1057, Fig. 2 (arXiv:2209.06833, CC BY 4.0): a Sun-like star around a 9.6 solar-mass black hole, P = 186 d, e = 0.45; grey lines are posterior draws.</div>

## Lab 05: the same tools on a binary star

* your own star, simulated like Gaia BH1 (Day 1), in `data/<netid>.csv`
    * about 900 days of radial velocities in three seasons
    * only one star's spectrum is seen (a **single-lined** binary), on an **eccentric** orbit
* seven parameters: mean velocity $\gamma$, amplitude $K$, period $P$, eccentricity $e$, orientation $\omega$, time of closest approach $T_p$, and a jitter $s$
    * the jitter enters the likelihood exactly as today: $\sigma_i^2 + s^2$, with its $\ln(\sigma_i^2 + s^2)$ term
* Part 1 finds candidate periods with a periodogram, walked through in the lab (the full story Oct 20/22)
* compute the mass function $f(M)$ for every sample
    * that histogram is $f(M)$'s posterior; it decides if your companion is a black hole
* due **Wed Oct 7 by Noon**; Abha's office hours are Wed 11 to noon on Zoom

## Before you go

* **Lab 05** is posted, due Wed Oct 7 by Noon (fork then PR, as always); every date is in `ASSIGNMENTS.md`
* Tue Oct 6, Gautham:
    * the Bayes factor for choosing between models (moved from Sep 29)
    * convergence across several chains, and samplers that follow the gradient (Hamiltonian Monte Carlo, `PyMC`)
* Nov 3/5: simulation-based inference, when you can simulate data but cannot write the likelihood
* the **midterm** posts Thu Oct 8, due Wed Oct 14 by Noon
* **Quiz 3** is tentatively Tue Oct 20 and will cover Bayes and MCMC